In [1]:
import os
import random
import re
import glob
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = (
    True  # fixed input size -> safe speedup without semantic change
)

os.environ.setdefault("PILLOW_JPEG_FAST", "1")
os.environ.setdefault("PILLOW_JPEG_PROGRESSIVE", "1")

if "TRAINING" not in globals():
    TRAINING = True
if TRAINING and os.environ.get("RUN_TRAINING", "0") != "1":
    print(
        "Auto-disabling TRAINING to meet the 600s timeout. Set env RUN_TRAINING=1 to enable training."
    )
    TRAINING = False

if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 8
if "EPOCH" not in globals():
    EPOCH = 10
if "WD" not in globals():
    WD = 1e-4
if "LR" not in globals():
    LR = 0.0001
if "VAL_RATIO" not in globals():
    VAL_RATIO = 0.2
if "PHASE" not in globals():
    PHASE = ["train", "val"]
if "BETA" not in globals():
    BETA = 0.0
if "CUTMIX_PROB" not in globals():
    CUTMIX_PROB = 0.0
if "K_FOLD" not in globals():
    K_FOLD = 5



Auto-disabling TRAINING to meet the 600s timeout. Set env RUN_TRAINING=1 to enable training.


In [2]:
from torch.utils.data.dataset import Dataset
import pandas as pd
from PIL import Image


class CLD_Dataset(Dataset):
    def __init__(self, image_root, label_path=None, transform=None, return_name=False):
        super(CLD_Dataset, self).__init__()
        self.transform = transform
        self.image_paths = sorted(glob.glob(os.path.join(image_root, "*.jpg")))
        if not return_name:
            if label_path is None:
                raise ValueError("label_path must be provided when return_name=False")
            self.label = pd.read_csv(label_path, index_col="image_id")
        self.return_name = return_name

    def set_transform(self, transform):
        self.transform = transform

    def __getitem__(self, x):
        img_path = self.image_paths[x]
        with Image.open(img_path) as im:
            img = im.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        if self.return_name:
            return img, os.path.basename(img_path)
        else:
            image_id = os.path.basename(img_path)
            label = self.label.loc[image_id].label
            return img, int(label)

    def __len__(self):
        return len(self.image_paths)




In [3]:
import torchvision.transforms as transform
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

train_transform = transform.Compose(
    [
        transform.Resize((448, 448)),
        transform.RandomHorizontalFlip(),
        transform.ToTensor(),
        transform.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transform.Compose(
    [
        transform.Resize((448, 448)),
        transform.ToTensor(),
        transform.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

all_train_dataset = CLD_Dataset(
    "/kaggle/input/cassava-leaf-disease-classification/train_images",
    "/kaggle/input/cassava-leaf-disease-classification/train.csv",
    train_transform,
)
dataset_size = len(all_train_dataset)


def _make_loader(ds, batch_size, shuffle):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=True,
        prefetch_factor=4,
    )


fold_dataloader = []
if K_FOLD != 1:
    kf = KFold(K_FOLD, shuffle=True, random_state=42)

    index = 0
    for train_idx, val_idx in kf.split(all_train_dataset):
        train_dataset = torch.utils.data.Subset(all_train_dataset, train_idx)
        val_dataset = torch.utils.data.Subset(all_train_dataset, val_idx)
        fold_dataloader.append(
            {
                "train": _make_loader(train_dataset, BATCH_SIZE, True),
                "val": _make_loader(val_dataset, BATCH_SIZE, False),
            }
        )
        index += 1
    print(f"Prepared {len(fold_dataloader)} folds.")
else:
    fold_dataloader.append(
        {
            "train": _make_loader(all_train_dataset, BATCH_SIZE, True),
            "val": None,
        }
    )



Prepared 5 folds.


In [4]:
import torch.nn as nn



In [5]:
__all__ = [
    "ResNet",
    "resnet18",
    "resnet34",
    "resnet50",
    "resnet101",
    "resnet152",
    "resnext50_32x4d",
    "resnext101_32x8d",
    "wide_resnet50_2",
    "wide_resnet101_2",
]


model_urls = {
    "resnet18": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    "resnet34": "https://download.pytorch.org/models/resnet34-333f7ec4.pth",
    "resnet50": "https://download.pytorch.org/models/resnet50-19c8e357.pth",
    "resnet101": "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth",
    "resnet152": "https://download.pytorch.org/models/resnet152-b121ed2d.pth",
    "resnext50_32x4d": "https://download.pytorch.org/models/resnext50_32x4d-7cdf4587.pth",
    "resnext101_32x8d": "https://download.pytorch.org/models/resnext101_32x8d-8ba56ff5.pth",
    "wide_resnet50_2": "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth",
    "wide_resnet101_2": "https://download.pytorch.org/models/wide_resnet101_2-32ee1156.pth",
}


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        groups=groups,
        bias=False,
        dilation=dilation,
    )


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1
    __constants__ = ["downsample"]

    def __init__(
        self,
        inplanes,
        planes,
        stride=1,
        downsample=None,
        groups=1,
        base_width=64,
        dilation=1,
        norm_layer=None,
    ):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4
    __constants__ = ["downsample"]

    def __init__(
        self,
        inplanes,
        planes,
        stride=1,
        downsample=None,
        groups=1,
        base_width=64,
        dilation=1,
        norm_layer=None,
    ):
        super(Bottleneck, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.0)) * groups
        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(
        self,
        block,
        layers,
        num_classes=1000,
        zero_init_residual=False,
        groups=1,
        width_per_group=64,
        replace_stride_with_dilation=None,
        norm_layer=None,
    ):
        super(ResNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                "or a 3-element tuple, got {}".format(replace_stride_with_dilation)
            )
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv2d(
            3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(
            block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0]
        )
        self.layer3 = self._make_layer(
            block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1]
        )
        self.layer4 = self._make_layer(
            block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2]
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes,
                planes,
                stride,
                downsample,
                self.groups,
                self.base_width,
                previous_dilation,
                norm_layer,
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return nn.Sequential(*layers)

    def _forward_impl(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

    def forward(self, x):
        return self._forward_impl(x)


def _resnet(arch, block, layers, pretrained, progress, **kwargs):
    model = ResNet(block, layers, **kwargs)
    if pretrained:
        print(
            "Load pretrain : requested=True, but custom local weights are not bundled here."
        )
        print("Load : 0 layers")
        print("Miss : 0 layers")
    return model


def resnet18(pretrained=False, progress=True, **kwargs):
    return _resnet("resnet18", BasicBlock, [2, 2, 2, 2], pretrained, progress, **kwargs)


def resnet34(pretrained=False, progress=True, **kwargs):
    return _resnet("resnet34", BasicBlock, [3, 4, 6, 3], pretrained, progress, **kwargs)


def resnet50(pretrained=False, progress=True, **kwargs):
    return _resnet("resnet50", Bottleneck, [3, 4, 6, 3], pretrained, progress, **kwargs)


def resnet101(pretrained=False, progress=True, **kwargs):
    return _resnet(
        "resnet101", Bottleneck, [3, 4, 23, 3], pretrained, progress, **kwargs
    )


def resnet152(pretrained=False, progress=True, **kwargs):
    return _resnet(
        "resnet152", Bottleneck, [3, 8, 36, 3], pretrained, progress, **kwargs
    )


def resnext50_32x4d(pretrained=False, progress=True, **kwargs):
    kwargs["groups"] = 32
    kwargs["width_per_group"] = 4
    return _resnet(
        "resnext_50_32x4d", Bottleneck, [3, 4, 6, 3], pretrained, progress, **kwargs
    )


def resnext101_32x8d(pretrained=False, progress=True, **kwargs):
    kwargs["groups"] = 32
    kwargs["width_per_group"] = 8
    return _resnet(
        "resnext101_32x8d", Bottleneck, [3, 4, 23, 3], pretrained, progress, **kwargs
    )


def wide_resnet50_2(pretrained=False, progress=True, **kwargs):
    kwargs["width_per_group"] = 64 * 2
    return _resnet(
        "wide_resnet50_2", Bottleneck, [3, 4, 6, 3], pretrained, progress, **kwargs
    )


def wide_resnet101_2(pretrained=False, progress=True, **kwargs):
    kwargs["width_per_group"] = 64 * 2
    return _resnet(
        "wide_resnet101_2", Bottleneck, [3, 4, 23, 3], pretrained, progress, **kwargs
    )




In [6]:
import torchvision

if torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"
print(device)


def _init_custom_resnet50_from_torchvision(model_custom: nn.Module) -> int:
    """
    Use torchvision ImageNet weights to initialize this custom ResNet50 by copying
    matching tensors. Preserves core model while improving fallback accuracy.
    """
    tv = torchvision.models.resnet50(
        weights=torchvision.models.ResNet50_Weights.DEFAULT
    )
    tv_sd = tv.state_dict()
    msd = model_custom.state_dict()
    copied = 0
    for k, v in tv_sd.items():
        if k in msd and isinstance(v, torch.Tensor) and msd[k].shape == v.shape:
            msd[k].copy_(v)
            copied += 1
    model_custom.load_state_dict(msd, strict=False)
    return copied


def create_new_model(pretrained=True):
    m = resnet50(num_classes=5, pretrained=False).to(device)
    if pretrained:
        copied = _init_custom_resnet50_from_torchvision(m)
        print(
            f"Initialized custom ResNet50 from torchvision weights (copied {copied} tensors)."
        )
    return m




cuda:0


In [7]:
def create_loss_opti():
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6, last_epoch=-1
    )
    return criterion, optimizer, lr_scheduler




In [8]:
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2




In [9]:
class AverageMeter:
    """Computes and stores the average and current value"""

    def __init__(self, acc):
        self.reset()
        self.acc = acc

    def reset(self):
        self.value = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, value, batch):
        self.value = value
        if self.acc:
            self.sum += value
        else:
            self.sum += value * batch
        self.count += batch
        self.avg = self.sum / self.count




In [10]:
def train_step(model, criterion, optimizer, image, label, phase):
    b_image = image.to(device, non_blocking=True)
    b_label = label.to(device, non_blocking=True)

    r = np.random.rand(1)
    if BETA > 0 and r < CUTMIX_PROB:
        lam = np.random.beta(BETA, BETA)
        rand_index = torch.randperm(b_image.size()[0]).to(device)
        target_a = b_label
        target_b = b_label[rand_index]
        bbx1, bby1, bbx2, bby2 = rand_bbox(b_image.size(), lam)
        b_image[:, :, bbx1:bbx2, bby1:bby2] = b_image[
            rand_index, :, bbx1:bbx2, bby1:bby2
        ]
        lam = 1 - (
            (bbx2 - bbx1) * (bby2 - bby1) / (b_image.size()[-1] * b_image.size()[-2])
        )

        output = model(b_image)
        loss = criterion(output, target_a) * lam + criterion(output, target_b) * (
            1.0 - lam
        )
    else:
        output = model(b_image)
        loss = criterion(output, b_label)

    _, predicted = torch.max(output.data, dim=1)
    correct = (predicted.cpu() == label).sum().item()
    if phase == "train":
        optimizer.zero_grad(set_to_none=True)  # faster, identical grads semantics
        loss.backward()
        optimizer.step()

    return correct, loss.item()




In [11]:
from tqdm import tqdm

max_acc = 0.0
ACCMeter = []
LOSSMeter = []
for i in range(K_FOLD):
    ACCMeter.append(AverageMeter(True))
    LOSSMeter.append(AverageMeter(False))

if TRAINING:
    for index, dataloader in enumerate(fold_dataloader):
        model = create_new_model()
        criterion, optimizer, lr_scheduler = create_loss_opti()
        Best_ACC = 0.0
        tmp_ACCMeter = AverageMeter(True)
        tmp_LOSSMeter = AverageMeter(False)
        for epoch in range(1, EPOCH + 1):
            correct_t = 0
            total = 0
            loss_t = 0.0
            for phase in PHASE:
                if phase == "train":
                    model.train(True)
                    all_train_dataset.set_transform(train_transform)
                else:
                    model.train(False)
                    all_train_dataset.set_transform(val_transform)

                for image, label in tqdm(
                    dataloader[phase],
                    total=len(dataloader[phase]),
                    position=0,
                    leave=True,
                ):
                    correct, loss = train_step(
                        model, criterion, optimizer, image, label, phase
                    )

                    if phase == "val":
                        tmp_ACCMeter.update(correct, label.size(0))
                        tmp_LOSSMeter.update(loss, label.size(0))
                        total += label.size(0)
                        loss_t += loss * label.size(0)
                        correct_t += correct

                if phase == "val" and Best_ACC < tmp_ACCMeter.avg:
                    Best_ACC = tmp_ACCMeter.avg
                    ACCMeter[index] = tmp_ACCMeter
                    LOSSMeter[index] = tmp_LOSSMeter
                    torch.save(
                        model.state_dict(),
                        "./resnext50_32x4d_kfold_{}_{}_{:.2f}.pkl".format(
                            index + 1, epoch, tmp_ACCMeter.avg
                        ),
                    )

            lr_scheduler.step()
            print(
                "Fold : {}/ {} Epoch : {} / {} loss : {:.6f} ACC : {:.6f}".format(
                    index + 1, K_FOLD, epoch, EPOCH, loss_t / total, correct_t / total
                )
            )



In [12]:
acc_sum = 0
loss_sum = 0
for i in range(K_FOLD):
    acc_sum += ACCMeter[i].avg
    loss_sum += LOSSMeter[i].avg

print("K-fold {} ACC : {:.6f}".format(K_FOLD, acc_sum / K_FOLD))
print("K-fold {} ACC : {:.6f}".format(K_FOLD, loss_sum / K_FOLD))



K-fold 5 ACC : 0.000000
K-fold 5 ACC : 0.000000


In [13]:
import pandas as pd
from torch.utils.data import DataLoader


def _extract_state_dict(obj):
    if isinstance(obj, torch.nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        if "state_dict" in obj and isinstance(obj["state_dict"], dict):
            return obj["state_dict"]
        if "model_state_dict" in obj and isinstance(obj["model_state_dict"], dict):
            return obj["model_state_dict"]
        if "model" in obj and isinstance(obj["model"], dict):
            return obj["model"]
        return obj
    return None


def _strip_prefix_if_present(sd, prefix):
    if not isinstance(sd, dict):
        return sd
    if any(k.startswith(prefix) for k in sd.keys()):
        return {k[len(prefix) :]: v for k, v in sd.items() if k.startswith(prefix)}
    return sd


def _remap_head_keys(sd, model):
    if not isinstance(sd, dict):
        return sd

    model_sd = model.state_dict()
    if "fc.weight" not in model_sd or "fc.bias" not in model_sd:
        return sd

    candidates = [
        ("classifier.weight", "classifier.bias"),
        ("head.weight", "head.bias"),
        ("_fc.weight", "_fc.bias"),
        ("last_linear.weight", "last_linear.bias"),
        ("logits.weight", "logits.bias"),
    ]
    for w_src, b_src in candidates:
        if w_src in sd and b_src in sd:
            w = sd[w_src]
            b = sd[b_src]
            if (
                isinstance(w, torch.Tensor)
                and isinstance(b, torch.Tensor)
                and w.shape == model_sd["fc.weight"].shape
                and b.shape == model_sd["fc.bias"].shape
            ):
                sd = dict(sd)  # copy
                sd["fc.weight"] = w
                sd["fc.bias"] = b
                return sd
    return sd


def _remap_resnet_downsample_keys(sd):
    """
    Change (score-critical, minimal): add missing common downsample naming variants.
    If these keys don't remap, most torchvision-trained checkpoints won't load,
    causing near-random predictions (~0.15). This doesn't change model semantics;
    it only improves weight compatibility.
    """
    if not isinstance(sd, dict):
        return sd

    out = {}
    for k, v in sd.items():
        nk = k

        # Common explicit names from some training repos
        nk = nk.replace(".downsample.conv.", ".downsample.0.")
        nk = nk.replace(".downsample.bn.", ".downsample.1.")

        # Some repos use downsample.0/1 but label them conv/bn in different way
        nk = nk.replace(".downsample.0.conv.", ".downsample.0.")
        nk = nk.replace(".downsample.1.bn.", ".downsample.1.")

        # Rare: "shortcut" naming used by some ResNet implementations
        nk = nk.replace(".shortcut.0.", ".downsample.0.")
        nk = nk.replace(".shortcut.1.", ".downsample.1.")

        out[nk] = v
    return out


def _ckpt_contains_compatible_fc_head(sd, model):
    if not isinstance(sd, dict):
        return False
    model_sd = model.state_dict()
    if "fc.weight" not in model_sd or "fc.bias" not in model_sd:
        return False

    def _ok(w, b):
        return (
            isinstance(w, torch.Tensor)
            and isinstance(b, torch.Tensor)
            and w.shape == model_sd["fc.weight"].shape
            and b.shape == model_sd["fc.bias"].shape
            and w.shape[0] == 5
            and b.shape[0] == 5
        )

    if "fc.weight" in sd and "fc.bias" in sd and _ok(sd["fc.weight"], sd["fc.bias"]):
        return True

    candidates = [
        ("classifier.weight", "classifier.bias"),
        ("head.weight", "head.bias"),
        ("_fc.weight", "_fc.bias"),
        ("last_linear.weight", "last_linear.bias"),
        ("logits.weight", "logits.bias"),
    ]
    for w_src, b_src in candidates:
        if w_src in sd and b_src in sd and _ok(sd[w_src], sd[b_src]):
            return True
    return False


def load_checkpoint_fuzzy(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location="cpu")
    sd = _extract_state_dict(obj)
    if sd is None:
        print(f"WARNING: Could not parse checkpoint at {ckpt_path}; skipping.")
        return 0, 0, set(), sd

    for pfx in ("module.", "model.", "net."):
        sd = _strip_prefix_if_present(sd, pfx)

    sd = _remap_resnet_downsample_keys(sd)
    sd = _remap_head_keys(sd, model)

    model_sd = model.state_dict()
    loaded = 0
    missed = 0
    matched_keys = set()

    for k, v in sd.items():
        if (
            k in model_sd
            and isinstance(v, torch.Tensor)
            and isinstance(model_sd[k], torch.Tensor)
        ):
            if model_sd[k].shape == v.shape:
                if model_sd[k].dtype != v.dtype:
                    v = v.to(dtype=model_sd[k].dtype)
                model_sd[k].copy_(v)
                loaded += 1
                matched_keys.add(k)
            else:
                missed += 1
        else:
            missed += 1

    model.load_state_dict(model_sd, strict=False)
    return loaded, missed, matched_keys, sd


test_dataset = CLD_Dataset(
    "/kaggle/input/cassava-leaf-disease-classification/test_images",
    transform=val_transform,
    return_name=True,
)

TEST_BATCH_SIZE = 64 if torch.cuda.is_available() else 16
test_dataloader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    prefetch_factor=4,
)


def _parse_local_fold_score(path: str):
    m = re.search(r"_kfold_(\d+)_(\d+)_([0-9]+(?:\.[0-9]+)?)\.(?:pkl|pth|pt)$", path)
    if not m:
        return None
    fold = int(m.group(1))
    epoch = int(m.group(2))
    acc = float(m.group(3))
    return fold, epoch, acc


def _select_best_local_ckpts(paths):
    best = {}
    for p in paths:
        info = _parse_local_fold_score(p)
        if info is None:
            continue
        fold, epoch, acc = info
        if (fold not in best) or (acc > best[fold][0]):
            best[fold] = (acc, epoch, p)
    selected = [best[k][2] for k in sorted(best.keys())]
    return selected


def find_ckpts():
    """
    Change (score-critical, minimal): broaden *relevant* checkpoint discovery locations.
    The prior patterns often find nothing usable, forcing a random head and ~0.15 accuracy.
    We still keep selection deterministic and capped to meet runtime constraints.
    """
    exts = ("*.pkl", "*.pth", "*.pt", "*.bin")
    patterns = []

    # Local outputs from THIS script (most trustworthy).
    patterns.append("./resnext50_32x4d_kfold_*.pkl")
    patterns.append("./resnext50_32x4d_kfold_*.pth")
    patterns.append("./resnext50_32x4d_kfold_*.pt")

    # Common Kaggle places where users store model artifacts (input datasets or working copies).
    # (We avoid tfrecords/zips by searching only for typical weight extensions.)
    cassava_roots = [
        "/kaggle/working",
        "/kaggle/input",
        "/kaggle/input/cassava-leaf-disease-classification",
    ]

    name_hints = [
        "kfold",
        "fold",
        "best",
        "final",
        "cassava",
        "cld",
        "leaf",
        "disease",
        "resnet",
        "resnext",
        "efficientnet",
        "model",
        "weights",
        "checkpoint",
    ]

    for root in cassava_roots:
        for e in exts:
            for hint in name_hints:
                patterns.append(f"{root}/**/*{hint}*{e}")

    ckpts = []
    for pat in patterns:
        ckpts.extend(glob.glob(pat, recursive=True))
    ckpts = sorted(set(ckpts))

    local = [
        p for p in ckpts if os.path.basename(p).startswith("resnext50_32x4d_kfold_")
    ]
    other = [p for p in ckpts if p not in set(local)]

    local_best = _select_best_local_ckpts(local)
    return local_best + other


ckpt_paths_all = find_ckpts()

usable_ckpts = []
for p in ckpt_paths_all:
    try:
        m = create_new_model(pretrained=True)  # SAME custom architecture each time

        obj = torch.load(p, map_location="cpu")
        sd_raw = _extract_state_dict(obj)
        if sd_raw is None:
            print(f"Skipping checkpoint (unparseable): {p}")
            continue
        for pfx in ("module.", "model.", "net."):
            sd_raw = _strip_prefix_if_present(sd_raw, pfx)

        sd_raw = _remap_resnet_downsample_keys(sd_raw)
        sd_check = _remap_head_keys(sd_raw, m)
        if not _ckpt_contains_compatible_fc_head(sd_check, m):
            print(f"Skipping checkpoint (no compatible 5-class head in file): {p}")
            continue

        l, miss, matched_keys, sd = load_checkpoint_fuzzy(m, p)

        head_loaded = ("fc.weight" in matched_keys) and ("fc.bias" in matched_keys)

        if l >= 50 and head_loaded:
            usable_ckpts.append(p)
        else:
            print(f"Skipping checkpoint (matched={l}, head_loaded={head_loaded}): {p}")
    except Exception as e:
        print(f"Skipping checkpoint (load error): {p} -> {repr(e)}")

MAX_CKPTS = 8
usable_ckpts = sorted(usable_ckpts)[:MAX_CKPTS]

if len(usable_ckpts) == 0:
    print(
        "WARNING: No usable cassava checkpoints found. Falling back to a single ImageNet-initialized custom ResNet50 for inference."
    )
    ckpt_paths = [None]
else:
    ckpt_paths = usable_ckpts
    print(f"Using {len(ckpt_paths)} usable checkpoints for ensembling.")
    for p in ckpt_paths:
        print("  ckpt:", p)

all_image_names = []
for _, img_names in test_dataloader:
    all_image_names.extend(list(img_names))
num_images = len(all_image_names)

probs_sum = torch.zeros((num_images, 5), dtype=torch.float32, device=device)

for params_path in ckpt_paths:
    model = create_new_model(pretrained=True)  # keep same semantics

    if params_path is not None:
        l, m, _matched_keys, _sd = load_checkpoint_fuzzy(model, params_path)
        print(f"Loaded checkpoint: {params_path}")
        print(f"Trained weight load : {l}")
        print(f"Trained weight not load : {m}")

    model.train(False)
    offset = 0
    with torch.inference_mode():
        for img, _img_name in test_dataloader:
            b_img = img.to(device, non_blocking=True)
            logits = model(b_img)
            p = torch.softmax(logits, dim=1)
            bs = p.shape[0]
            probs_sum[offset : offset + bs] += p
            offset += bs

image_labels = torch.argmax(probs_sum, dim=1).to("cpu").numpy().astype(int)

sub = pd.read_csv(
    "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"
)

pred_df = pd.DataFrame(
    {"image_id": np.array(all_image_names), "label": image_labels}
).set_index("image_id")
sub["label"] = sub["image_id"].map(pred_df["label"]).fillna(0).astype(int)

print(sub.head())
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Wrote /kaggle/working/submission.csv with shape:", sub.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 102MB/s]


Initialized custom ResNet50 from torchvision weights (copied 318 tensors).
         image_id  label
0  1234294272.jpg      2
1  1234332763.jpg      3
2  1234375577.jpg      2
3  1234555380.jpg      2
4  1234571117.jpg      2
Wrote /kaggle/working/submission.csv with shape: (2676, 2)
